# Hands-On Machine Learning Book Assistant - Kaggle RAG Pipeline

This Kaggle-ready Core Track notebook turns the attached *Hands-On Machine Learning* book into a cited RAG assistant. It covers extraction, cleaning, segmentation, embeddings, persisted Chroma retrieval, local notebook generation, and evaluation.

**Before running:** Create a Kaggle Dataset containing the book PDF, attach it to this notebook, and set `KAGGLE_DATASET_SLUG` below to the folder name shown under `/kaggle/input/`. Do not upload copyrighted material unless you have the right to use it.

## 0. Kaggle setup

In Kaggle, enable **Internet** in Notebook options for the first run. It allows the notebook to download the embedding and generation models. If Internet must remain off, attach the required Hugging Face models as Kaggle Datasets and replace their names below with their `/kaggle/input/...` paths.

The install cell is written for Kaggle. It uses a Hugging Face model for the notebook demonstration because Kaggle does not provide a reliable local Ollama server. Your local FastAPI backend can still use Ollama later with the same Chroma collection and prompt.

In [ ]:
# Kaggle package setup - run once per Kaggle session.
!pip install -q pypdf chromadb sentence-transformers transformers accelerate tqdm

import json
import re
import shutil
from pathlib import Path
from typing import Any

import chromadb
import pandas as pd
import torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from transformers import pipeline

pd.set_option("display.max_colwidth", 180)

## 1. Kaggle configuration

Set `KAGGLE_DATASET_SLUG` to the exact dataset folder name visible in Kaggle's right sidebar after attaching your book dataset. The code uses 900-character chunks with 150-character overlap, balancing focused retrieval against loss of context at boundaries.

Kaggle files in `/kaggle/input/` are read-only. All generated outputs go to `/kaggle/working/`, which Kaggle makes available in the notebook Output section when you save a version.

In [ ]:
# Change this to the exact folder name shown in /kaggle/input/ after you attach your dataset.
KAGGLE_DATASET_SLUG = "book-hands-on-machine-learning"

PROJECT_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
DATA_DIR = INPUT_ROOT / KAGGLE_DATASET_SLUG
VECTOR_STORE_DIR = PROJECT_ROOT / "vector_store"
CONFIG_DIR = PROJECT_ROOT / "rag_output"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GENERATION_MODEL = "google/flan-t5-base"
COLLECTION_NAME = "hands_on_ml_book"
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150
TOP_K = 4
REBUILD_VECTOR_STORE = True

if not DATA_DIR.exists():
    available = [path.name for path in INPUT_ROOT.iterdir()] if INPUT_ROOT.exists() else []
    raise FileNotFoundError(
        f"Dataset folder not found: {DATA_DIR}. Attach your dataset, then set KAGGLE_DATASET_SLUG. "
        f"Currently attached folders: {available}"
    )

for folder in (VECTOR_STORE_DIR, CONFIG_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print(f"Book dataset: {DATA_DIR}")
print(f"Kaggle outputs: {PROJECT_ROOT}")

## 2. Load and inspect the book dataset

Each record preserves its filename and page number. This metadata flows through the pipeline and becomes the citation displayed with each final answer. Files that cannot be parsed are listed so you can identify PDFs that require OCR.

> The book PDF should be text-selectable. This notebook cannot extract text from image-only/scanned pages without an OCR step.

In [ ]:
SUPPORTED_SUFFIXES = {".pdf", ".txt", ".md", ".csv"}


def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def read_pdf(path: Path) -> list[dict[str, Any]]:
    records = []
    for page_number, page in enumerate(PdfReader(path).pages, start=1):
        text = clean_text(page.extract_text() or "")
        if text:
            records.append({"source": path.name, "page": page_number, "text": text})
    return records


def read_text(path: Path) -> list[dict[str, Any]]:
    text = clean_text(path.read_text(encoding="utf-8", errors="ignore"))
    return [{"source": path.name, "page": 1, "text": text}] if text else []


def read_csv(path: Path) -> list[dict[str, Any]]:
    frame = pd.read_csv(path).fillna("").astype(str)
    return [{"source": path.name, "page": 1, "text": clean_text(frame.to_csv(index=False))}]


def load_corpus(data_dir: Path) -> tuple[list[dict[str, Any]], list[dict[str, str]]]:
    files = sorted(path for path in data_dir.rglob("*") if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES)
    if not files:
        raise FileNotFoundError("No PDF, TXT, MD, or CSV files found in the attached Kaggle dataset.")

    records, failures = [], []
    for path in files:
        try:
            if path.suffix.lower() == ".pdf":
                records.extend(read_pdf(path))
            elif path.suffix.lower() in {".txt", ".md"}:
                records.extend(read_text(path))
            else:
                records.extend(read_csv(path))
        except Exception as exc:
            failures.append({"file": path.name, "reason": str(exc)})
    return records, failures


documents, failed_files = load_corpus(DATA_DIR)
documents_df = pd.DataFrame(documents)
if documents_df.empty:
    raise ValueError("No extractable text was found. Use a text-based PDF or add OCR before ingestion.")

print(f"Extracted {len(documents_df)} page/record(s) from {documents_df.source.nunique()} file(s).")
print("Formats:", sorted({Path(source).suffix.lower() for source in documents_df.source}))
display(documents_df[["source", "page", "text"]].head())
display(pd.DataFrame(failed_files) if failed_files else pd.DataFrame([{"status": "No files failed to parse."}]))

## 3. Segment the dataset into overlapping chunks

The splitter favors paragraph, sentence, and word boundaries. For each chunk, it stores the source, page, chunk index, and character range, giving the backend enough metadata to provide grounded citations.

In [ ]:
def split_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[tuple[str, int, int]]:
    if not 0 <= overlap < chunk_size:
        raise ValueError("CHUNK_OVERLAP must be non-negative and smaller than CHUNK_SIZE.")

    chunks, start, length = [], 0, len(text)
    while start < length:
        end = min(start + chunk_size, length)
        if end < length:
            boundary = max(text.rfind("\n\n", start, end), text.rfind(". ", start, end), text.rfind(" ", start, end))
            if boundary > start + chunk_size // 2:
                end = boundary + (1 if text[boundary:boundary + 2] == ". " else 0)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append((chunk, start, end))
        if end >= length:
            break
        start = max(end - overlap, start + 1)
    return chunks


chunk_records = []
for row in documents_df.itertuples(index=False):
    for chunk_index, (chunk_text, char_start, char_end) in enumerate(split_text(row.text)):
        chunk_records.append({
            "chunk_id": f"{Path(row.source).stem}-p{row.page}-c{chunk_index}",
            "source": row.source,
            "page": int(row.page),
            "chunk_index": chunk_index,
            "char_start": char_start,
            "char_end": char_end,
            "text": chunk_text,
        })

chunks_df = pd.DataFrame(chunk_records)
if chunks_df.empty:
    raise ValueError("No chunks were created. Check the text extraction results.")

print(f"Created {len(chunks_df)} chunks; mean length: {chunks_df.text.str.len().mean():.0f} characters.")
display(chunks_df[["chunk_id", "source", "page", "char_start", "char_end", "text"]].head(5))

## 4. Create embeddings and persist the vector store

Chroma saves embeddings in `/kaggle/working/vector_store/`. When you save a Kaggle version, download the generated ZIP from the Output section and copy it into your local FastAPI project. The backend should load it once at startup, never rebuild it on a user request.

In [ ]:
embedder = SentenceTransformer(EMBEDDING_MODEL)
client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))

if REBUILD_VECTOR_STORE:
    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

BATCH_SIZE = 64
for start in tqdm(range(0, len(chunks_df), BATCH_SIZE), desc="Embedding chunks"):
    batch = chunks_df.iloc[start:start + BATCH_SIZE]
    embeddings = embedder.encode(batch.text.tolist(), normalize_embeddings=True, show_progress_bar=False).tolist()
    collection.upsert(
        ids=batch.chunk_id.tolist(),
        documents=batch.text.tolist(),
        embeddings=embeddings,
        metadatas=[
            {"source": row.source, "page": int(row.page), "chunk_index": int(row.chunk_index)}
            for row in batch.itertuples(index=False)
        ],
    )

print(f"Persisted {collection.count()} chunks in {VECTOR_STORE_DIR}")

## 5. Retrieve chunks and build a grounded prompt

The model receives only retrieved chunks. It is instructed to refuse unsupported answers and cite the source-page label for each factual claim.

In [ ]:
def retrieve(question: str, top_k: int = TOP_K) -> list[dict[str, Any]]:
    if not question.strip():
        raise ValueError("The question cannot be blank.")
    result = collection.query(
        query_embeddings=embedder.encode([question], normalize_embeddings=True).tolist(),
        n_results=min(top_k, collection.count()),
    )
    return [
        {
            "chunk_id": result["ids"][0][i],
            "text": result["documents"][0][i],
            "source": result["metadatas"][0][i]["source"],
            "page": result["metadatas"][0][i]["page"],
            "distance": float(result["distances"][0][i]),
        }
        for i in range(len(result["ids"][0]))
    ]


def build_prompt(question: str, passages: list[dict[str, Any]]) -> str:
    context = "\n\n".join(
        f"[{item['source']} | page {item['page']} | {item['chunk_id']}]\n{item['text']}"
        for item in passages
    )
    return f"""You are a helpful assistant for the Hands-On Machine Learning book.
Answer only from the supplied book context. If the context is insufficient, say:
'I do not have enough information in the provided book passages.'
Use concise prose and cite every factual claim with the supplied bracketed source label.

Context:
{context}

Question: {question}
Answer:"""


# Use device=0 on Kaggle GPU; CPU is used automatically when no accelerator is selected.
generator = pipeline(
    "text2text-generation",
    model=GENERATION_MODEL,
    device=0 if torch.cuda.is_available() else -1,
)


def generate_answer(question: str, passages: list[dict[str, Any]]) -> str:
    response = generator(
        build_prompt(question, passages),
        max_new_tokens=220,
        do_sample=False,
        truncation=True,
    )
    return response[0]["generated_text"].strip()


def ask(question: str, top_k: int = TOP_K) -> dict[str, Any]:
    passages = retrieve(question, top_k)
    return {
        "question": question,
        "answer": generate_answer(question, passages),
        "sources": sorted({f"{item['source']} (page {item['page']})" for item in passages}),
        "passages": passages,
    }


sample_question = "What is the difference between supervised and unsupervised learning?"
display(pd.DataFrame(retrieve(sample_question))[["source", "page", "distance", "text"]])

## 6. Test the complete RAG flow

The first run downloads `google/flan-t5-base`. Select a Kaggle GPU accelerator for faster generation. This model is for Kaggle experimentation; use the same `build_prompt()` plus Ollama in the final local backend required by the assignment.

In [ ]:
result = ask(sample_question)
print(result["answer"])
print("\nSources:", ", ".join(result["sources"]))

## 7. Evaluate at least 10 book questions

These questions target core material from *Hands-On Machine Learning*. Review each answer manually and complete `relevant_context` and `grounded`; retrieval distance alone does not prove that an answer is correct.

In [ ]:
EVALUATION_QUESTIONS = [
    "What is machine learning?",
    "What is the difference between supervised and unsupervised learning?",
    "What are the four main categories of machine learning systems?",
    "What are common challenges of machine learning?",
    "What is the difference between a training set, validation set, and test set?",
    "What is overfitting and how can it be reduced?",
    "What is underfitting and how can it be addressed?",
    "Why is feature scaling important for gradient descent?",
    "What is the difference between batch gradient descent and stochastic gradient descent?",
    "What is regularization and why is it useful?",
]

evaluation_rows = []
for question in tqdm(EVALUATION_QUESTIONS, desc="Evaluating"):
    response = ask(question)
    evaluation_rows.append({
        "question": question,
        "retrieved_source": "; ".join(response["sources"]),
        "answer": response["answer"],
        "relevant_context": "",  # Manually enter yes/no after review.
        "grounded": "",          # Manually enter yes/no after review.
        "notes": "",
    })

evaluation_df = pd.DataFrame(evaluation_rows)
evaluation_path = CONFIG_DIR / "evaluation_results.csv"
evaluation_df.to_csv(evaluation_path, index=False)
display(evaluation_df)
print(f"Review and complete: {evaluation_path}")

### Evaluation findings (complete after manual review)

Record the main failure cases you observe: for example, nearby-but-incorrect retrieval, a question needing more than `TOP_K` passages, or a vague query. Typical mitigations are improving text cleanup, adjusting the chunking parameters, changing `TOP_K`, using a stronger embedding model, or strengthening the refusal instruction in the prompt.

## 8. Export Kaggle outputs for the backend

This cell saves the pipeline settings and compresses the persisted Chroma store. Save a Kaggle version, then download both artifacts from its Output section. Extract `vector_store.zip` into your FastAPI project's `data/vector_store/` directory.

In [ ]:
pipeline_config = {
    "embedding_model": EMBEDDING_MODEL,
    "generation_model_used_in_kaggle": GENERATION_MODEL,
    "backend_generation_model": "Set your local Ollama model in the backend .env, for example llama3.2:3b",
    "collection_name": COLLECTION_NAME,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "top_k": TOP_K,
    "vector_store_path": "data/vector_store",
}
config_path = CONFIG_DIR / "rag_config.json"
config_path.write_text(json.dumps(pipeline_config, indent=2), encoding="utf-8")

archive_path = shutil.make_archive(str(PROJECT_ROOT / "vector_store"), "zip", VECTOR_STORE_DIR)
print(f"Saved configuration: {config_path}")
print(f"Download this persisted vector store: {archive_path}")